## 0. Imports

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold

## 0. Paths Configuration

Set data paths, battle constants, feature sets, and model validation settings.

In [2]:
BASE_KAGGLE_PATH       = '/kaggle/input/competitions/fds-pokemon-battles-prediction-2026'
TRAIN_FILE_PATH        = BASE_KAGGLE_PATH +'/train.jsonl'
TEST_FILE_PATH         = BASE_KAGGLE_PATH + '/test.jsonl'
TYPE_CHART_PATH        = BASE_KAGGLE_PATH + '/type_chart_gen2.csv'
SUBMISSION_PATH        = "/kaggle/working/"
Path(SUBMISSION_PATH).mkdir(exist_ok=True)
SAMPLE_SUBMISSION_PATH = BASE_KAGGLE_PATH + "/sample_submission.csv"

pd.set_option("display.max_columns", None)  # Show every feature column.
pd.set_option("display.width", None)

# 1. Features Sets

Set of features used for training and validation.

In [3]:
TEAM_SIZE = 6

STATISTICS = ["hp", "atk", "def", "spa", "spd", "spe"]
HP_PCT_LOST_INTERVALS = ["0_10", "10_20", "20_30"]
MULTIPLIERS = [0, 0.25, 0.5, 1, 2, 4]
STATUSES = ["slp", "frz", "tox", "psn", "par", "brn"]
BOOSTS = ["atk", "def", "spa", "spd", "spe"]

# From player 1 POV
EFFECTS = {
    "good": [
        "beatup", "bide", "destinybond", "endure", "focusband",
        "focusenergy", "lockon", "magnitude", "mimic", "mindreader",
        "mysteryberry", "protect", "quickclaw", "safeguard", "substitute",
        "typechange"
    ],
    "bad": [
        "attract", "bind", "clamp", "confusion", "curse",
        "disable", "encore", "firespin", "foresight", "futuresight",
        "leechseed", "nightmare", "perish", "pursuit", "spite",
        "trapped", "whirlpool", "wrap"
    ]
}
STAB = 1.5

# --- HELD ITEMS DICTIONARIES & METRIC PRIORS ---
JUNK_ITEMS = {
    "berryjuice", "berry", "noitem", "pokeball", "firestone",
    "energypowder", "aspearberry", "gengarite", "alakazite", "blackbelt"
}
STATUS_BERRIES = {
    "miracleberry", "mintberry", "przcureberry", "psncureberry",
    "bitterberry", "lumberry", "chestoberry"
}
TYPE_BOOST_ITEMS = {
    "charcoal", "magnet", "blackbelt", "pinkbow", "polkadotbow",
    "mysticwater", "hardstone", "poisonbarb", "softsand", "silverpowder",
    "blackglasses", "sharpbeak", "spelltag", "miracleseed", "twistedspoon",
    "nevermeltice", "dragonscale", "dragonfang", "metalcoat"
}
RNG_ITEMS = {
    "berserkgene", "scopelens", "quickclaw", "focusband", "brightpowder", "kingsrock"
}
HEALING_MOVES = {
    "rest", "recover", "softboiled", "milkdrink", "synthesis",
    "morningsun", "moonlight", "drainpunch", "megadrain", "gigadrain", "leechlife"
}
ITEM_GROUPS = {
    "leftovers": {"leftovers"},
    "junk": JUNK_ITEMS,
    "status_berry": STATUS_BERRIES,
    "type_boost": TYPE_BOOST_ITEMS,
    "rng": RNG_ITEMS,
}
ITEM_METRICS = [*ITEM_GROUPS, "non_leftovers"]
KNOWN_ITEMS = set().union(*ITEM_GROUPS.values(), {"thickclub", "lightball"})

DEPLOYED_METRICS = ["deployed_turns", "level", "remaining_effective_hp"]
DEPLOYED_SUMMARIES = [
    "deployed_count", "min_level", "underleveled_count", "deployed_underleveled_count",
    "deployed_min_lvl", "active_wmean_lvl", "lead_level", "rem_effective_hp_ratio",
]
TACTIC_MOVES = {
    "rest_used": {"rest"},
    "sleeptalk_used": {"sleeptalk"},
    "phaze_used": {"roar", "whirlwind"},
}
TACTIC_METRICS = ["moves_used", "switches", "eff_power", *TACTIC_MOVES, "spikes_active"]

# --- FIXED FEATURES ---
FEATURES_DANI = [
    # 1. Reconstructed Headcount & Bulk
    "p1_fnt",
    "p2_confirmed_kos",
    "p2_estimated_true_kos",
    "diff_estimated_pokemon_alive",
    "p2_known_remaining_hp_sum",
    "diff_mean_remaining_hp_pct",
    "p1_mean_remaining_hp_pct",
    "p2_mean_remaining_hp_pct",
    # 2. HP Intervals & Loss (Includes Turns 0-10 Early Tempo)
    "diff_hp_pct_lost_0_10",
    "p1_hp_pct_lost_0_10",
    "p2_hp_pct_lost_0_10",
    "diff_hp_pct_lost_10_20",
    "diff_hp_pct_lost_20_30",
    "p1_hp_pct_lost_20_30",
    "p2_hp_pct_lost_20_30",
    "diff_mean_hp_pct_lost",
    # 3. Statuses (Unweighted Totals)
    "diff_sum_remaining_statuses",
    "p1_sum_remaining_statuses",
    "p2_sum_remaining_statuses",
    "diff_sum_statuses",
    # 4. Stat Stages & Boosts (Unweighted Totals)
    "diff_sum_remaining_boosts",
    "p1_sum_remaining_boosts",
    "p2_sum_remaining_boosts",
    "diff_sum_boosts",
    "p1_sum_boosts",
    "p2_sum_boosts",
    # 5. Type Effectiveness & Real Attack Multipliers
    "p1_sum_mult",
    "p2_sum_mult",
    "diff_0_mult",
    "diff_2_mult",
    "diff_sum_mult",
    "diff_mean_mult",
    "diff_sum_effects",
    "diff_sum_remaining_effects",
    "p1_sum_remaining_effects",
    # 6. Core Defensive Base Stats & Differentials
    "diff_mean_def",
    "diff_mean_hp",
    "p1_mean_def",
    "p2_mean_hp",
    "diff_mean_atk",
    "diff_mean_spa",
    "diff_mean_spd",
    "diff_mean_spe",
    "diff_mean_bst",
    # 7. Held Items & Detections
    "p1_item_leftovers_count",
    "p1_item_has_all_leftovers",
    "p1_item_unique_count",
    "p1_item_junk_count",
    "p1_item_status_berry_count",
    "p1_item_type_boost_count",
    "p1_item_rng_count",
    "p1_has_thickclub_marowak",
    "p2_detected_leftovers_count",
    "diff_leftovers_count",
    "diff_junk_items_count",
    # 8. Deployed Counts, Mean Level & Effective Bulk
    "p1_mean_level",
    "p1_deployed_count",
    "p2_deployed_count",
    "diff_deployed_count",
    "p1_rem_effective_hp_ratio",
    "p2_rem_effective_hp_ratio",
    "diff_rem_effective_hp_ratio",
    # 9. Tactical Tempo, Hazards & RestTalk
    "diff_moves_used",
    "diff_switches",
    "p1_eff_power_mean",
    "p2_eff_power_mean",
    "diff_eff_power_mean",
    "diff_rest_used",
    "diff_sleeptalk_used",
    "diff_spikes_active",
    "diff_phaze_used",
    # 10. Macro Team Typing & Defensive Profiling
    "diff_type_diversity",
    "diff_average_weakness_severity",
    "diff_shared_weakness_type_count",
    "p2_known_type_count",
    "p2_detected_rng_count",
]

FEATURES_FIL = [
    # 1. Team Health & Reconstructed Headcount
    "diff_mean_remaining_hp_pct",
    "p1_mean_remaining_hp_pct",
    "p2_mean_remaining_hp_pct",
    "p1_fnt",                          
    "p2_confirmed_kos",               
    "p2_estimated_true_kos",           
    "diff_estimated_pokemon_alive",    
    "p2_known_remaining_hp_sum",       
    # 2. HP Loss
    "diff_hp_pct_lost_20_30",
    "p1_hp_pct_lost_20_30",
    "p2_hp_pct_lost_20_30",
    "diff_hp_pct_lost_10_20",
    "p1_hp_pct_lost_10_20",
    "p2_hp_pct_lost_10_20",
    "diff_hp_pct_lost_0_10",
    # 3. Statuses
    "diff_sum_remaining_statuses",
    "p1_sum_remaining_statuses",
    "p2_sum_remaining_statuses",
    "diff_sum_statuses",
    # 4. Stat Stages & Boosts
    "diff_sum_remaining_boosts",
    "p1_sum_remaining_boosts",
    "p2_sum_remaining_boosts",
    "diff_sum_boosts",
    "p1_sum_boosts",
    "p2_sum_boosts",
    # 5. Type Effectiveness & Battle Effects
    "p1_sum_mult",
    "p2_sum_mult",
    "diff_0_mult",
    "diff_2_mult",
    "diff_sum_mult",
    "diff_mean_mult",
    "diff_sum_effects",
    "diff_sum_remaining_effects",
    "p1_sum_remaining_effects",
    # 6. Core Defensive Base Stats
    "diff_mean_def",
    "diff_mean_hp",
    "p1_mean_def",
    "p2_mean_hp",
    # 7. Held Items
    "p1_item_leftovers_count",
    "p1_item_has_all_leftovers",
    "p1_item_unique_count",
    "p1_item_junk_count",
    "p1_item_status_berry_count",
    "p1_item_type_boost_count",
    "p1_item_rng_count",
    "p1_has_thickclub_marowak",
    "p2_detected_leftovers_count",
    "diff_leftovers_count",
    "diff_junk_items_count",
    # 8. Deployed Counts, Mean Level & Effective HP Ratio
    "p1_mean_level",  
    "p1_deployed_count",
    "p2_deployed_count",
    "diff_deployed_count",
    "p1_rem_effective_hp_ratio",
    "p2_rem_effective_hp_ratio",
    "diff_rem_effective_hp_ratio",
    # 9. Tactical Tempo, Effective Power, Hazards & RestTalk
    "diff_moves_used",
    "diff_switches",
    "p1_eff_power_mean",
    "p2_eff_power_mean",
    "diff_eff_power_mean",
    "diff_rest_used",
    "diff_sleeptalk_used",
    "diff_spikes_active",
    "diff_phaze_used",

    "p1_hp_pct_lost_0_10", 
    "p2_hp_pct_lost_0_10",
    "diff_mean_atk", 
    "diff_mean_spa", 
    "diff_mean_spd", 
    "diff_mean_spe", 
    "diff_mean_bst",
    "p2_known_type_count",
    "p2_sum_remaining_effects", 
    "p1_type_diversity", 
    "p2_type_diversity", 
    "diff_type_diversity",
    "p1_average_weakness_severity", 
    "p2_average_weakness_severity", 
    "diff_average_weakness_severity",
    "p1_shared_weakness_type_count", 
    "p2_shared_weakness_type_count", 
    "diff_shared_weakness_type_count",
    "p2_detected_rng_count",
]

FEATURES_LORE = [
    # BASE STATS
    "p1_sum_def",
    "p1_sum_spe",
    "p1_min_base_atk",
    "p1_min_base_def",
    "p1_min_base_spa",
    "p1_max_base_hp",
    "p1_max_base_atk",
    "p1_std_base_hp",
    "p1_std_base_atk",
    "p1_std_base_spa",
    "p2_sum_hp",
    "p2_sum_atk",
    "p2_sum_spa",
    "p2_sum_spd",
    "p2_mean_def",
    "p2_max_base_spe",
    "p2_std_base_atk",
    "p2_std_base_spe",
    "p1_sum_bst",
    "p2_sum_bst",
    "diff_sum_hp",
    "diff_sum_atk",
    "diff_sum_def",
    "diff_sum_spa",
    "diff_sum_spd",
    "diff_mean_spa",
    "diff_sum_bst",

    # HP PCT LOST
    "p1_hp_pct_lost_20_30",
    "p1_sum_hp_lost",
    "p1_std_hp_pct_lost",
    "p2_hp_pct_lost_0_10",
    "p2_hp_pct_lost_10_20",
    "p2_hp_pct_lost_20_30",
    "p2_sum_hp_lost",
    "p2_std_hp_pct_lost",
    "diff_hp_pct_lost_0_10",
    "diff_hp_pct_lost_10_20",
    "diff_hp_pct_lost_20_30",
    "diff_sum_hp_pct_lost",
    "diff_std_hp_pct_lost",

    # TEAM TYPE
    "p2_known_type_count",
    "diff_average_weakness_severity",

    # MULTIPLIERS
    "p1_0.5_mult",
    "p1_1_mult",
    "diff_0_mult",
    "diff_0.5_mult",
    "p1_count_mult",
    "p2_count_mult",
    "p1_std_mult",
    "diff_sum_mult",
    "diff_mean_mult",
    "diff_min_mult",
    "diff_max_mult",

    # STATUSES
    "p2_sum_statuses",
    "diff_sum_statuses",

    # BOOSTS
    "p1_spd_boost",

    # EFFECTS
    "diff_sum_bad_effects",

    # LAST-SEEN SNAPSHOT
    "p1_1_pkm_remaining_hp_pct",
    "p1_1_pkm_remaining_status",
    "p1_3_pkm_remaining_hp_pct",
    "p1_4_pkm_remaining_hp_pct",
    "p1_4_pkm_remaining_status",
    "p2_0_pkm_remaining_hp_pct",
    "p2_1_pkm_remaining_hp_pct",
    "p2_3_pkm_remaining_status",
    "p2_remaining_psn",
    "diff_remaining_slp",
    "diff_remaining_par",
    "p2_remaining_atk_boost",
    "p2_remaining_def_boost",
    "diff_remaining_atk_boost",
    "p1_sum_remaining_boosts",
    "p2_sum_remaining_statuses",
    "p2_sum_remaining_boosts",
    "diff_sum_remaining_statuses",
    "diff_sum_remaining_boosts",

    # FAINT EVIDENCE
    "p1_sum_confirmed_kos",
    "p2_std_estimated_true_kos",
    "diff_sum_confirmed_kos",
    "diff_sum_estimated_true_kos",
    "diff_std_confirmed_kos",
    "diff_std_estimated_true_kos",

    # HELD ITEMS (P2 VALUES ARE OBSERVED EVIDENCE)
    "p2_0_pkm_item_leftovers",
    "p2_sum_item_leftovers",

    # DEPLOYED POKEMON AND LEVELS
    "p1_0_pkm_deployed_turns",
    "p1_1_pkm_deployed_turns",
    "p1_1_pkm_remaining_effective_hp",
    "p1_5_pkm_remaining_effective_hp",
    "p2_1_pkm_deployed_turns",
    "p2_1_pkm_remaining_effective_hp",
    "p2_2_pkm_remaining_effective_hp",
    "p2_4_pkm_deployed_turns",
    "p2_5_pkm_deployed_turns",
    "p2_5_pkm_remaining_effective_hp",
    "p1_sum_remaining_effective_hp",
    "p1_std_deployed_turns",
    "p2_sum_remaining_effective_hp",
    "p2_std_deployed_turns",
    "diff_sum_remaining_effective_hp",
    "diff_std_deployed_turns",
    "diff_std_remaining_effective_hp",
    "p1_rem_effective_hp_ratio",
    "p2_deployed_count",
    "p2_rem_effective_hp_ratio",
    "diff_rem_effective_hp_ratio",

    # TACTICAL TEMPO, POWER AND HAZARDS
    "p1_sum_eff_power",
    "p1_sum_spikes_active",
    "p1_std_switches",
    "p1_std_eff_power",
    "p2_sum_eff_power",
    "p2_sum_spikes_active",
    "p2_std_eff_power",
    "p2_std_spikes_active",
    "diff_sum_spikes_active",
    "diff_std_moves_used",
    "diff_std_eff_power",
    "diff_std_spikes_active",
    "p1_spikes_active",
    "diff_spikes_active",
]

## 1.2 Models Configuration

In [4]:
MODEL_CONFIGS = {
    "lr_dani":  ("lr",       FEATURES_DANI, SUBMISSION_PATH + "submission_lr_dani.csv"),
    "lr_fil":   ("lr",       FEATURES_FIL,  SUBMISSION_PATH + "submission_lr_fil.csv"),
    "ens_lore": ("ensemble", FEATURES_LORE, SUBMISSION_PATH + "submission_ens_lore.csv"),
}

ENSEMBLE_N_ESTIMATORS = 150
ENSEMBLE_WEIGHTS = [1.0, 1.0, 1.0, 1.0]  # lgb, xgb, cat, lr
RANDOM_STATE = 42
K_FOLDS = 5
N_REPEATS = 3
DECISION_THRESHOLD = 0.5

# 2. Load Data

Read the training and test JSONL files into lists of battle records.

In [5]:
def load_data(file_path: str) -> list:
    """
    Loads the whole `file_path` and stores it as a list of battles
    """

    print(f"Loading data from '{file_path}'...")
    data = []
    try:
        with open(file_path, "r") as f:
            for line in f:
                data.append(json.loads(line))
        print(f"SUCCESS: loaded {len(data)} battles.")
        
    except FileNotFoundError:
        print(f"ERROR: could not find the file at '{file_path}'.")

    return data


# Loading data
train_data = load_data(TRAIN_FILE_PATH)
test_data  = load_data(TEST_FILE_PATH)

Loading data from '/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/train.jsonl'...
SUCCESS: loaded 10000 battles.
Loading data from '/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/test.jsonl'...
SUCCESS: loaded 5000 battles.


## 2.1 Create and Load Pokemon Stats

Build a species lookup of base stats and types from the available team details.

In [6]:
def create_stats_map(train_data: list, test_data: list) -> dict:
    """
    Build a species stats/type lookup from supplied battle details
    """

    print("Creating Pokémon stats lookup...")
    stats_map = {}
    for data in (train_data, test_data):
        for battle in data:
            for pokemon in [*battle["p1_team_details"], battle["p2_lead_details"]]:
                stats_map[pokemon["name"]] = {
                    "base_stats": {f"base_{stat}": pokemon[f"base_{stat}"] for stat in STATISTICS},
                    "types": pokemon["types"],
                }
    print(f"SUCCESS: found {len(stats_map)} pokemon.")
    return stats_map

## 2.2 Create and Load Pokemon Types

Load and normalize the Generation II type chart to calculate attack effectiveness against both defensive types.

In [7]:

def normalize_type_chart(type_chart: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize type-chart column names and type values in place
    """

    # Clear columns headers
    type_chart.columns = type_chart.columns.astype(str).str.strip()

    # Clear columns values
    for col in ["attacking_type", "defending_type"]:
        # If a value is Nan break the run
        if type_chart[col].isna().any():
            nan_rows = type_chart[type_chart[col].isna()].index.tolist()
            raise ValueError(f"ERROR: found NaN value in column '{col}' at rows {nan_rows}.")

        # Remove white spaces and lower
        type_chart[col] = type_chart[col].astype(str).str.strip().str.lower()

    print(f"SUCCESS: cleaned {len(type_chart)} entries.")
    return type_chart

def load_type_chart(file_path: str) -> pd.DataFrame | None:
    """
    Read a type-chart CSV and return its normalized DataFrame
    """

    print(f"Loading data from '{file_path}'...")
    try:
        raw_df = pd.read_csv(file_path)
        return normalize_type_chart(raw_df)

    except FileNotFoundError:
        print(f"ERROR: could not find the file at '{file_path}'.")

# Getting type charts
type_chart   = load_type_chart(TYPE_CHART_PATH)
df_indexed   = type_chart.set_index(["attacking_type", "defending_type"])
type_chart   = df_indexed["multiplier"].to_dict()
ATTACK_TYPES = {attack_type for attack_type, _ in type_chart}
stats_map    = create_stats_map(train_data, test_data)

Loading data from '/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/type_chart_gen2.csv'...
SUCCESS: cleaned 110 entries.
Creating Pokémon stats lookup...
SUCCESS: found 247 pokemon.


# 3. Extract Features

Create one feature row per battle using the same extraction logic for training and test data. The features cover:

- **Team strength:** base-stat summaries, type diversity, and shared defensive weaknesses.
- **Battle health:** HP loss over turns 1–10, 11–20, and 21–30, latest observed HP, and confirmed or estimated knockouts.
- **Matchups and conditions:** move effectiveness, status counts, stat boosts, and beneficial or harmful effects.
- **Items and deployment:** known P1 items, detected P2 item evidence, active turns, levels, and estimated remaining effective HP.
- **Tactics:** move and switch rates, effective attack power, Spikes, Rest/Sleep Talk use, and forced switches.

Include per-Pokémon values, team summaries, and differences between players. Opponent features combine observed evidence with estimates where information is incomplete.

In [8]:
# --- BASE STATS ---
def get_stats(turn: dict, p2_stats: dict) -> None:
    """
    Store player 2's active pokemon stats in p2_stats if found in stats_map
    """

    pkm2_name = turn["p2_pokemon_state"]["name"]
    if pkm2_name in stats_map:
        stats = stats_map[pkm2_name]["base_stats"]
        p2_stats[pkm2_name] = stats

def feature_base_stats(battle: dict, p2_stats: dict, features: dict) -> None:
    """
    Add base-stat for P1's full team and P2's known team.
    For each stat, compute sum, mean, min, max and population standard deviation
    """

    for player, team in (("p1", battle["p1_team_details"]), ("p2", p2_stats.values())):
        # Single values + merged values
        features[f"{player}_sum_bst"] = 0
        features[f"{player}_mean_bst"] = 0
        bst_values = np.zeros(len(team))  # One total per Pokemon with known stats
        for stat in STATISTICS:
            values = [pokemon[f"base_{stat}"] for pokemon in team]
            features[f"{player}_sum_{stat}"]  = np.sum(values)
            features[f"{player}_mean_{stat}"] = np.mean(values)
            features[f"{player}_min_base_{stat}"] = min(values)
            features[f"{player}_max_base_{stat}"] = max(values)
            features[f"{player}_std_base_{stat}"] = float(np.std(values, ddof=0))

            features[f"{player}_sum_bst"] += features[f"{player}_sum_{stat}"]
            bst_values += values

        # Merged values
        features[f"{player}_mean_bst"] = features[f"{player}_sum_bst"] / TEAM_SIZE
        features[f"{player}_std_base_bst"] = float(np.std(bst_values, ddof=0))

    # Differentials
    for stat in STATISTICS:
        features[f"diff_sum_{stat}"]  = features[f"p1_sum_{stat}"] - features[f"p2_sum_{stat}"]
        features[f"diff_mean_{stat}"] = features[f"p1_mean_{stat}"] - features[f"p2_mean_{stat}"]
    features["diff_sum_bst"]  = features["p1_sum_bst"]  - features["p2_sum_bst"]
    features["diff_mean_bst"] = features["p1_mean_bst"] - features["p2_mean_bst"]
    features["diff_std_base_bst"] = features["p1_std_base_bst"] - features["p2_std_base_bst"]


# --- TEAM TYPES ---
def feature_team_types(battle: dict, p2_stats: dict, features: dict) -> None:
    """
    Add known team type diversity and repeated defensive weaknesses
    """
    
    p1_team = [pokemon["name"] for pokemon in battle["p1_team_details"]]
    for player, team in (("p1", p1_team), ("p2", p2_stats)):
        team_types = []
        unique_types = set()
        for name in team:
            # If name is present then we collect the types related
            types = [type for type in stats_map.get(name, {}).get("types", [])]
            if types:
                # Ensure first 2 entries
                team_types.append(types)
                unique_types.update(types)

        # Number of pokemons with known types
        known_count = len(team_types)
        features[f"{player}_known_type_count"] = known_count
        features[f"{player}_type_diversity"] = len(unique_types)
        
        total_weakness_severity, shared_weakness_type_count = 0.0, 0
        for attack_type in ATTACK_TYPES:
            multipliers = [get_multiplier(attack_type, *types) for types in team_types]
            # Sum of (super-)effective moves off team based on attact_type
            total_weakness_severity += sum((multiplier // 2) if multiplier > 1.0 else 0.0 for multiplier in multipliers)
            # At least two pokemon are weak to the same type (multiplier > 1.0)
            shared_weakness_type_count += sum(multiplier > 1.0 for multiplier in multipliers) >= 2

        features[f"{player}_average_weakness_severity"] = total_weakness_severity / known_count if known_count else np.nan
        features[f"{player}_shared_weakness_type_count"] = shared_weakness_type_count if known_count else np.nan

    features["diff_type_diversity"] = features["p1_type_diversity"] - features["p2_type_diversity"]
    features["diff_average_weakness_severity"]  = features["p2_average_weakness_severity"] - features["p1_average_weakness_severity"]
    features["diff_shared_weakness_type_count"] = features["p2_shared_weakness_type_count"] - features["p1_shared_weakness_type_count"]

# --- MULTPLIERS --- 
def get_multiplier(move_type: str, pkm_type1: str, pkm_type2: str) -> float:
    """
    Return attack effectiveness against both defensive types
    """

    m1 = type_chart.get((move_type, pkm_type1), 1.0) if pkm_type1 else 1.0
    m2 = type_chart.get((move_type, pkm_type2), 1.0) if pkm_type2 else 1.0
    return m1 * m2

# --- HP LOSS ---
def get_hp_pct_lost(turn: dict, p1_hp_map: dict, p2_hp_map: dict) -> None:
    """
    Accumulate each observed Pokemon's net HP loss.
    Track its latest HP fraction across switches, assuming full HP before its first observation.
    Add previous HP minus current HP to the corresponding 1-10, 11-20 or 21-30 turn interval.
    Healing contributes negative loss
    """

    for player, hp_map in (("p1", p1_hp_map), ("p2", p2_hp_map)):
        name = turn[f"{player}_pokemon_state"]["name"]
        hp = turn[f"{player}_pokemon_state"]["hp_pct"]
        if name not in hp_map:
            hp_map[name] = {"last_hp": 1.0, "losses": [0.0, 0.0, 0.0]}
        pokemon = hp_map[name]
        pokemon["losses"][(turn["turn"] - 1) // 10] += pokemon["last_hp"] - hp
        pokemon["last_hp"] = hp

def feature_hp_lost(p1_hp_map: dict, p2_hp_map: dict, features: dict) -> None:
    """
    Add interval losses, total losses and per-Pokemon loss dispersion.
    Interval values and totals are sums of HP fractions.
    Compute population standard deviation from each Pokemon's total net loss, padding unseen team members with zero loss.
    Also compute differences P2 - P1 loss
    """

    for player, team in (("p1", p1_hp_map), ("p2", p2_hp_map)):
        # Single values + merged values
        features[f"{player}_sum_hp_lost"] = 0
        # Keep one total per Pokemon, with zero loss for unseen Pokemon.
        hp_lost_values = np.zeros(TEAM_SIZE)
        for i, interval in enumerate(HP_PCT_LOST_INTERVALS):
            values = [pokemon["losses"][i] for pokemon in team.values()]
            features[f"{player}_hp_pct_lost_{interval}"] = sum(values)
            features[f"{player}_sum_hp_lost"] += features[f"{player}_hp_pct_lost_{interval}"]
            hp_lost_values[:len(team)] += values

        # Merged values
        features[f"{player}_mean_hp_pct_lost"] = features[f"{player}_sum_hp_lost"] / TEAM_SIZE
        features[f"{player}_std_hp_pct_lost"] = float(np.std(hp_lost_values, ddof=0))

    # Differentials
    for interval in HP_PCT_LOST_INTERVALS:
        features[f"diff_hp_pct_lost_{interval}"] = features[f"p2_hp_pct_lost_{interval}"] - features[f"p1_hp_pct_lost_{interval}"]
    features["diff_sum_hp_pct_lost"]  = features["p2_sum_hp_lost"] - features["p1_sum_hp_lost"]
    features["diff_mean_hp_pct_lost"] = features["p2_mean_hp_pct_lost"] - features["p1_mean_hp_pct_lost"]
    features["diff_std_hp_pct_lost"]  = features["p2_std_hp_pct_lost"] - features["p1_std_hp_pct_lost"]


# --- TYPE EFFECTIVENESS ---
def get_mults(turn: dict, p1_mults: dict, p2_mults: dict) -> None:
    """
    Count the effectiveness of each move.
    Skip absent moves and status moves.
    Unknown P2 species use neutral defensive types
    """

    mv1 = turn["p1_move_details"]
    if mv1 is not None and mv1["category"] != "status":
        mv1_type = mv1["type"]
        pkm2_name = turn["p2_pokemon_state"]["name"]
        pkm2_stats = stats_map.get(pkm2_name)
        if pkm2_stats is not None:
            pkm2_types = pkm2_stats.get("types", ["notype", "notype"])
        else:
            pkm2_types = ["notype", "notype"]
        mult1 = get_multiplier(mv1_type, pkm2_types[0], pkm2_types[1])
        p1_mults[mult1] += 1

    mv2 = turn["p2_move_details"]
    if mv2 is not None and mv2["category"] != "status":
        mv2_type = mv2["type"]
        pkm1_name = turn["p1_pokemon_state"]["name"]
        pkm1_types = stats_map[pkm1_name]["types"]
        mult2 = get_multiplier(mv2_type, pkm1_types[0], pkm1_types[1])
        p2_mults[mult2] += 1

def feature_mults(p1_mults: dict, p2_mults: dict, features: dict) -> None:
    """
    Add multiplier bucket counts and summaries of recorded moves.

    Compute count, sum, min, max, mean and population standard deviation.
    Bucket differences use P2 - P1 below 1x and P1 - P2 at or above 1x.
    """

    for player, team in (("p1", p1_mults), ("p2", p2_mults)):
        # Single values + merged values
        values = []
        features[f"{player}_count_mult"] = 0
        features[f"{player}_sum_mult"] = 0
        for mult in MULTIPLIERS:
            value = team[mult]
            values.extend([mult] * value)
            features[f"{player}_{mult}_mult"] = value
            features[f"{player}_sum_mult"] += mult * value
            features[f"{player}_count_mult"] += value

        # Merged values
        features[f"{player}_min_mult"] = min(values) if values else np.nan
        features[f"{player}_max_mult"] = max(values) if values else np.nan
        features[f"{player}_mean_mult"] = features[f"{player}_sum_mult"] / features[f"{player}_count_mult"] if values else np.nan
        features[f"{player}_std_mult"] = float(np.std(values, ddof=0)) if values else np.nan

    # Differentials
    features["diff_count_mult"] = 0
    for mult in MULTIPLIERS:
        if mult < 1:
            features[f"diff_{mult}_mult"] = features[f"p2_{mult}_mult"] - features[f"p1_{mult}_mult"]
            features["diff_count_mult"] += features[f"diff_{mult}_mult"]
        else:
            features[f"diff_{mult}_mult"] = features[f"p1_{mult}_mult"] - features[f"p2_{mult}_mult"]
            features["diff_count_mult"] += features[f"diff_{mult}_mult"]
    features["diff_min_mult"] = features["p1_min_mult"] - features["p2_min_mult"]
    features["diff_max_mult"] = features["p1_max_mult"] - features["p2_max_mult"]
    features["diff_sum_mult"]  = features["p1_sum_mult"] - features["p2_sum_mult"]
    features["diff_mean_mult"] = features["p1_mean_mult"] - features["p2_mean_mult"]
    features["diff_std_mult"] = features["p1_std_mult"] - features["p2_std_mult"]


# --- STATUSES (NO FAINT) ---
def get_statuses(turn: dict, p1_status: dict, p2_status: dict) -> None:
    """
    Accumulate active-Pokemon status observations.
    Exclude nostatus and fnt.
    A status that persists across several observed
    turns contributes once on each turn
    """
    
    for player, team in (("p1", p1_status), ("p2", p2_status)):
        pkm_state = turn[f"{player}_pokemon_state"]["status"]
        if pkm_state != "nostatus" and pkm_state != "fnt":
            if pkm_state not in team:
                team[pkm_state] = 0
            team[pkm_state] += 1

def feature_statuses(p1_status_map: dict, p2_status_map: dict, n_turns: int, features: dict) -> None:
    """
    Add per-status counts, normalized fractions and status totals.
    Normalize each count and the total by the denominator n_turns.
    All status differences use P2 - P1
    """

    for player, team in (("p1", p1_status_map), ("p2", p2_status_map)):
        # Single values + merged values
        features[f"{player}_sum_statuses"] = 0
        for status in STATUSES:
            value = team.get(status, 0)
            features[f"{player}_{status}"] = value
            features[f"{player}_status_frac_{status}"] = value / n_turns
            features[f"{player}_sum_statuses"] += value

        # Merged values
        features[f"{player}_mean_statuses"] = features[f"{player}_sum_statuses"] / n_turns

    # Differentials
    for status in STATUSES:
        features[f"diff_{status}"] = features[f"p2_{status}"] - features[f"p1_{status}"]
        features[f"diff_status_frac_{status}"] = features[f"diff_{status}"] / n_turns
    features["diff_sum_statuses"]  = features["p2_sum_statuses"] - features["p1_sum_statuses"]
    features["diff_mean_statuses"] = features["p2_mean_statuses"] - features["p1_mean_statuses"]


# --- BOOSTS (CONSISTENT SCHEMA & DIFFERENTIALS) ---
def get_boosts(turn: dict, p1_boosts: dict, p2_boosts: dict) -> None:
    """
    Accumulate observed signed boost stages for both active Pokemon.
    Add each stat's current stage on every turn.
    Persistent boosts are counted repeatedly
    """

    for player, team in (("p1", p1_boosts), ("p2", p2_boosts)):
        for boost, value in turn[f"{player}_pokemon_state"]["boosts"].items():
            team[boost] = team.get(boost, 0) + value

def feature_boosts(p1_boosts: dict, p2_boosts: dict, n_turns: int, features: dict) -> None:
    """
    Add per-stat boost totals and signed aggregate summaries.
    Sum the supplied stage observations.
    Divide the total by the denominator n_turns.
    All boost differences use P1 - P2
    """

    for player, team in (("p1", p1_boosts), ("p2", p2_boosts)):
        # Single values + merged values
        features[f"{player}_sum_boosts"] = 0
        for boost in BOOSTS:
            value = team.get(boost, 0)
            features[f"{player}_{boost}_boost"] = value
            features[f"{player}_sum_boosts"] += value

        # Merged values
        features[f"{player}_mean_boosts"] = features[f"{player}_sum_boosts"] / n_turns

    # Differentials
    for boost in BOOSTS:
        features[f"diff_{boost}_boost"] = features[f"p1_{boost}_boost"] - features[f"p2_{boost}_boost"]
    features["diff_sum_boosts"]  = features["p1_sum_boosts"] - features["p2_sum_boosts"]
    features["diff_mean_boosts"] = features["p1_mean_boosts"] - features["p2_mean_boosts"]


# --- EFFECTS FEATURES ---
def get_effects(turn: dict, p1_effects: dict, p2_effects: dict) -> None:
    """
    Accumulate each active Pokemon's observed effect.
    Exclude noeffect.
    Persistent effects contribute repeatedly
    """

    for player, team in (("p1", p1_effects), ("p2", p2_effects)):
        for effect in turn[f"{player}_pokemon_state"]["effects"]:
            if effect != "noeffect":
                team[effect] = team.get(effect, 0) + 1

def feature_effects(p1_effects: dict, p2_effects: dict, n_turns: int, features: dict) -> None:
    """
    Add effect counts, good/bad totals and signed net summaries.
    Group effects as good or bad for the Pokemon carrying them.
    Net effects equal good counts minus bad counts.
    Normalize group and net totals by the denominator n_turns.
    Differences use P1 - P2 for good/net effects and P2 - P1 for bad effects
    """

    for player, team in (("p1", p1_effects), ("p2", p2_effects)):
        # Single values + merged values
        for group, effects in EFFECTS.items():
            features[f"{player}_sum_{group}_effects"] = 0
            for effect in effects:
                value = team.get(effect, 0)
                features[f"{player}_{effect}_effect"] = value
                features[f"{player}_sum_{group}_effects"] += value
            features[f"{player}_mean_{group}_effects"] = features[f"{player}_sum_{group}_effects"] / n_turns

        # Merged values: good effects minus bad effects
        features[f"{player}_sum_effects"]  = features[f"{player}_sum_good_effects"] - features[f"{player}_sum_bad_effects"]
        features[f"{player}_mean_effects"] = features[f"{player}_sum_effects"] / n_turns

    # Differentials for player 1 POV
    for group, effects in EFFECTS.items():
        first, second = ("p1", "p2") if group == "good" else ("p2", "p1")
        for effect in effects:
            features[f"diff_{effect}_effect"] = features[f"{first}_{effect}_effect"] - features[f"{second}_{effect}_effect"]
        features[f"diff_sum_{group}_effects"] = features[f"{first}_sum_{group}_effects"] - features[f"{second}_sum_{group}_effects"]
        features[f"diff_mean_{group}_effects"] = features[f"{first}_mean_{group}_effects"] - features[f"{second}_mean_{group}_effects"]
    features["diff_sum_effects"] = features["p1_sum_effects"] - features["p2_sum_effects"]
    features["diff_mean_effects"] = features["p1_mean_effects"] - features["p2_mean_effects"]


# --- TEAM SNAPSHOT AT LAST TURN ---
def get_last_seen(turn: dict, p1_last_seen: dict, p2_last_seen: dict) -> None:
    """
    Replace each observed Pokemon's latest HP/status.
    Keep the previous snapshots of switched-out Pokemon until they are observed again
    """

    for player, team in (("p1", p1_last_seen), ("p2", p2_last_seen)):
        pokemon = turn[f"{player}_pokemon_state"]
        team[pokemon["name"]] = {
            "hp_pct": pokemon["hp_pct"],
            "status": pokemon["status"],
        }

def feature_last_seen(last_turn: dict, p1_last_seen: dict, p2_last_seen: dict, features: dict) -> None:
    """
    Add latest team HP/status summaries and final active boosts/effects.
    Use each observed Pokemon's latest HP and status, which may be older for benched Pokemon.
    Assume unseen team members have full HP and no status.
    Exclude fnt from status counts.
    Boosts and effects come from the final active Pokemon only.
    Means divide by TEAM_SIZE.
    Pokemon slots follow first-observed order, followed by unseen team members.
    Differences use P1 - P2 for HP, boosts and good/net effects,
    and P2 - P1 for statuses and bad effects.
    """

    for player, team in (("p1", p1_last_seen), ("p2", p2_last_seen)):
        team = list(team.values())
        # Full health pokemon padding
        team.extend({"hp_pct": 1.0, "status": "nostatus"} for _ in range(TEAM_SIZE - len(team)))

        # Single values + merged values
        features[f"{player}_sum_remaining_hp_pct"] = 0
        features[f"{player}_sum_remaining_statuses"] = 0
        for status in STATUSES:
            features[f"{player}_remaining_{status}"] = 0
        hp_values = []
        for i, pokemon in enumerate(team):
            hp = pokemon["hp_pct"]
            hp_values.append(hp)
            features[f"{player}_{i}_pkm_remaining_hp_pct"] = hp
            features[f"{player}_sum_remaining_hp_pct"] += hp
            status = pokemon["status"]
            features[f"{player}_{i}_pkm_remaining_status"] = status
            if status in STATUSES:
                features[f"{player}_remaining_{status}"] += 1
                features[f"{player}_sum_remaining_statuses"] += 1

        # Merged values
        features[f"{player}_mean_remaining_hp_pct"] = features[f"{player}_sum_remaining_hp_pct"] / TEAM_SIZE
        features[f"{player}_std_remaining_hp_pct"] = float(np.std(hp_values, ddof=0))
        features[f"{player}_mean_remaining_statuses"] = features[f"{player}_sum_remaining_statuses"] / TEAM_SIZE

        active = last_turn[f"{player}_pokemon_state"]

        # Single values + merged values
        features[f"{player}_sum_remaining_boosts"] = 0
        for boost in BOOSTS:
            value = active["boosts"].get(boost, 0)
            features[f"{player}_remaining_{boost}_boost"] = value
            features[f"{player}_sum_remaining_boosts"] += value
        features[f"{player}_mean_remaining_boosts"] = features[f"{player}_sum_remaining_boosts"] / TEAM_SIZE

        # Single values + merged values
        for group, effects in EFFECTS.items():
            features[f"{player}_sum_remaining_{group}_effects"] = 0
            for effect in effects:
                # Whether the final active Pokemon currently has this effect.
                value = int(effect in active["effects"])
                features[f"{player}_remaining_{effect}_effect"] = value
                features[f"{player}_sum_remaining_{group}_effects"] += value
            features[f"{player}_mean_remaining_{group}_effects"] = features[f"{player}_sum_remaining_{group}_effects"] / TEAM_SIZE
        features[f"{player}_sum_remaining_effects"] = features[f"{player}_sum_remaining_good_effects"] - features[f"{player}_sum_remaining_bad_effects"]
        features[f"{player}_mean_remaining_effects"] = features[f"{player}_sum_remaining_effects"] / TEAM_SIZE

    # Differentials from player 1 POV
    features[f"diff_sum_remaining_hp_pct"]  = features[f"p1_sum_remaining_hp_pct"]  - features[f"p2_sum_remaining_hp_pct"]
    features[f"diff_mean_remaining_hp_pct"] = features[f"p1_mean_remaining_hp_pct"] - features[f"p2_mean_remaining_hp_pct"]
    features[f"diff_std_remaining_hp_pct"]  = features[f"p1_std_remaining_hp_pct"]  - features[f"p2_std_remaining_hp_pct"]

    for status in STATUSES:
        features[f"diff_remaining_{status}"] = features[f"p2_remaining_{status}"] - features[f"p1_remaining_{status}"]
    features[f"diff_sum_remaining_statuses"]  = features[f"p2_sum_remaining_statuses"]  - features[f"p1_sum_remaining_statuses"]
    features[f"diff_mean_remaining_statuses"] = features[f"p2_mean_remaining_statuses"] - features[f"p1_mean_remaining_statuses"]
    
    for boost in BOOSTS:
        features[f"diff_remaining_{boost}_boost"] = features[f"p1_remaining_{boost}_boost"] - features[f"p2_remaining_{boost}_boost"]
    features[f"diff_sum_remaining_boosts"]  = features[f"p1_sum_remaining_boosts"]  - features[f"p2_sum_remaining_boosts"]
    features[f"diff_mean_remaining_boosts"] = features[f"p1_mean_remaining_boosts"] - features[f"p2_mean_remaining_boosts"]

    for group, effects in EFFECTS.items():
        first, second = ("p1", "p2") if group == "good" else ("p2", "p1")
        for effect in effects:
            features[f"diff_remaining_{effect}_effect"] = features[f"{first}_remaining_{effect}_effect"] - features[f"{second}_remaining_{effect}_effect"]
        features[f"diff_sum_remaining_{group}_effects"]  = features[f"{first}_sum_remaining_{group}_effects"]  - features[f"{second}_sum_remaining_{group}_effects"]
        features[f"diff_mean_remaining_{group}_effects"] = features[f"{first}_mean_remaining_{group}_effects"] - features[f"{second}_mean_remaining_{group}_effects"]
        features[f"diff_sum_remaining_effects"]  = features[f"p1_sum_remaining_effects"]  - features[f"p2_sum_remaining_effects"]
        features[f"diff_mean_remaining_effects"] = features[f"p1_mean_remaining_effects"] - features[f"p2_mean_remaining_effects"]


# --- PLAYER 2 FAINT ---
def get_p2_faint_evidence(turn: dict, idx: int, timeline: list, tracker: dict) -> None:
    """
    Track confirmed P2 KOs and likely KOs at a Pokemon's final observed departure.
    Low-HP departures are evidence only; they do not confirm a KO
    """

    p1, p2 = (turn[f"{player}_pokemon_state"] for player in ("p1", "p2"))
    p1_move, p2_move = (turn.get(f"{player}_move_details") or {} for player in ("p1", "p2"))
    name, hp, status, effects = p2["name"], p2["hp_pct"], p2["status"], p2["effects"]

    if status == "fnt" or hp <= 0 or p2_move.get("name") in {"explosion", "selfdestruct"}:
        tracker["confirmed"].add(name)
        tracker["ghost"].discard(name)

    # The next active Pokemon identifies a departure in the recorded timeline.
    if (
        idx + 1 >= len(timeline)
        or name in tracker["confirmed"]
        or any(t["p2_pokemon_state"]["name"] == name for t in timeline[idx + 1:])
        or p1_move.get("name") in {"roar", "whirlwind"}
    ):
        return

    damaging = p1_move.get("base_power", 0) > 0 and p1_move.get("category") in {"physical", "special"}
    residual = hp <= 0.15 and (status in {"tox", "psn", "brn"} or "leechseed" in effects)
    destiny_bond = "destinybond" in effects or "destinybond" in p1["effects"]
    if residual or (destiny_bond and damaging) or (damaging and hp <= 0.40):
        tracker["ghost"].add(name)

def feature_p2_faint_evidence(tracker: dict, p1_last_seen: dict, p2_last_seen: dict, features: dict) -> None:
    """
    Add per-Pokemon KO flags and team sums, means and population standard deviations.
    P1 KOs come from observed faint/zero-HP states; P2 also includes inferred KOs.
    Slots follow first-observed order, with zero KO evidence for unseen Pokemon.
    KO differences use P2 - P1, equivalent to P1 - P2 for Pokemon still alive
    """

    for player, team in (("p1", p1_last_seen), ("p2", p2_last_seen)):
        confirmed = {name for name, pokemon in team.items() if pokemon["status"] == "fnt" or pokemon["hp_pct"] <= 0}
        if player == "p2":
            confirmed |= tracker["confirmed"]
        estimated = confirmed | tracker["ghost"] if player == "p2" else confirmed
        names = list(team) + [None] * (TEAM_SIZE - len(team))

        for metric, fainted in (("confirmed_kos", confirmed), ("estimated_true_kos", estimated)):
            values = [int(name in fainted) for name in names]
            for i, value in enumerate(values):
                features[f"{player}_{i}_pkm_{metric}"] = value
            for summary, value in (("sum", np.sum(values)), ("mean", np.mean(values)), ("std", np.std(values, ddof=0))):
                features[f"{player}_{summary}_{metric}"] = float(value)
            features[f"{player}_{metric}"] = features[f"{player}_sum_{metric}"]

        features[f"{player}_fnt"] = features[f"{player}_confirmed_kos"]
        features[f"{player}_estimated_pokemon_alive"] = TEAM_SIZE - features[f"{player}_estimated_true_kos"]
        features[f"{player}_known_remaining_hp_sum"] = sum(pokemon["hp_pct"] for pokemon in team.values()) + max(0, TEAM_SIZE - len(team))

    for metric in ("confirmed_kos", "estimated_true_kos"):
        for summary in ("sum", "mean", "std"):
            features[f"diff_{summary}_{metric}"] = features[f"p2_{summary}_{metric}"] - features[f"p1_{summary}_{metric}"]
        features[f"diff_{metric}"] = features[f"p2_{metric}"] - features[f"p1_{metric}"]
    features["diff_estimated_pokemon_alive"] = features["p1_estimated_pokemon_alive"] - features["p2_estimated_pokemon_alive"]


# --- HELD ITEMS ---
def get_items(turn: dict, p2_item_tracker: dict) -> None:
    """
    Collect P2 item evidence by Pokemon, counting repeated activations only once.
    A small HP gain on consecutive active turns suggests Leftovers, unless a healing
    move or Leech Seed can explain it. Explicit item effects are recorded directly
    """

    state = turn["p2_pokemon_state"]
    name, hp = state["name"], state["hp_pct"]
    move = turn.get("p2_move_details") or {}
    items = p2_item_tracker["seen"].setdefault(name, set())
    healing = move.get("name") in HEALING_MOVES
    if name == p2_item_tracker.get("prev_mon"):
        hp_gain = hp - p2_item_tracker["prev_hp"]
        if (
            0 < hp_gain <= 1 / 16 + 0.01
            and not (healing or p2_item_tracker.get("prev_healing", False))
            and "leechseed" not in turn["p1_pokemon_state"]["effects"]
        ):
            items.add("leftovers")

    for effect in state["effects"]:
        if effect in KNOWN_ITEMS:
            items.add(effect)
    p2_item_tracker.update(prev_mon=name, prev_hp=hp, prev_healing=healing)

def feature_items(battle: dict, p2_item_tracker: dict, features: dict) -> None:
    """
    Add per-Pokemon items and item-group indicators, with team sum/mean/std and diffs.
    P1 slots use team order; P2 slots use first-observed order and unknown padding.
    P2 zeros mean no detected item evidence, rather than confirmed absence of an item.
    Item-group differences use P1 - P2, matching the original item counts
    """

    p1_items = [{pokemon.get("item") or "noitem"} for pokemon in battle["p1_team_details"]]
    for player, team in (("p1", p1_items), ("p2", list(p2_item_tracker["seen"].values()))):
        team = team + [set() for _ in range(TEAM_SIZE - len(team))]
        for i, items in enumerate(team):
            features[f"{player}_{i}_pkm_item"] = "|".join(sorted(items)) if items else "unknown"

        for group in ITEM_METRICS:
            values = [
                int(bool(items) and "leftovers" not in items) if group == "non_leftovers"
                else int(bool(items & ITEM_GROUPS[group]))
                for items in team
            ]
            for i, value in enumerate(values):
                features[f"{player}_{i}_pkm_item_{group}"] = value
            for summary, value in (("sum", np.sum(values)), ("mean", np.mean(values)), ("std", np.std(values, ddof=0))):
                features[f"{player}_{summary}_item_{group}"] = float(value)
            features[f"{player}_item_{group}_count"] = features[f"{player}_sum_item_{group}"]

        features[f"{player}_item_has_all_leftovers"] = int(features[f"{player}_sum_item_leftovers"] == TEAM_SIZE)
        features[f"{player}_item_unique_count"] = len(set().union(*team))
        features[f"{player}_item_non_leftovers"] = features[f"{player}_sum_item_non_leftovers"]

    for group in ITEM_METRICS:
        for summary in ("sum", "mean", "std"):
            features[f"diff_{summary}_item_{group}"] = features[f"p1_{summary}_item_{group}"] - features[f"p2_{summary}_item_{group}"]
    for group in ("leftovers", "rng"):
        features[f"p2_detected_{group}_count"] = features[f"p2_sum_item_{group}"]
    for name, group in (("leftovers_count", "leftovers"), ("junk_items_count", "junk")):
        features[f"diff_{name}"] = features[f"diff_sum_item_{group}"]
    for name, species, item in (("thickclub_marowak", {"marowak", "cubone"}, "thickclub"), ("lightball_pikachu", {"pikachu"}, "lightball")):
        features[f"p1_has_{name}"] = int(any(pokemon["name"] in species and pokemon.get("item") == item for pokemon in battle["p1_team_details"]))


# --- DEPLOYED POKEMON, LEVELS & EFFECTIVE HP ---
def calc_eff_stat(base_val: float, level: int, is_hp: bool = False) -> float:
    """
    Estimate a Gen 2 stat from base stat and level, assuming max DV and StatExp
    """

    core = ((base_val + 47) * 2 * level) / 100.0
    return core + level + 10 if is_hp else core + 5

def get_deployed_turns(turn: dict, p1_deployed_turns: dict, p2_deployed_turns: dict) -> None:
    """
    Count active turns per observed Pokemon for both players.
    """

    for player, team in (("p1", p1_deployed_turns), ("p2", p2_deployed_turns)):
        name = turn[f"{player}_pokemon_state"]["name"]
        team[name] = team.get(name, 0) + 1

def feature_deployed_and_levels(battle: dict, p1_deployed_turns: dict, p2_deployed_turns: dict, p1_hp_map: dict, p2_hp_map: dict, features: dict) -> None:
    """
    Add per-Pokemon deployment, level and remaining effective HP, plus sum/mean/std.
    P1 slots follow team order; P2 slots follow first-observed order, then its known
    lead if unseen, then unknown Pokemon. P2 uses its known lead level and assumes
    level 100 otherwise. Unknown species use base HP 80; unseen slots use 85/full HP.
    Differences use P1 - P2. Deployment counts have no fixed good/bad meaning
    """

    p1_team = {pokemon["name"]: pokemon for pokemon in battle["p1_team_details"]}
    p2_lead = battle["p2_lead_details"]
    p2_names = list(dict.fromkeys([*p2_deployed_turns, p2_lead["name"]]))
    for player, names, deployed_turns, hp_map in (
        ("p1", list(p1_team), p1_deployed_turns, p1_hp_map),
        ("p2", p2_names, p2_deployed_turns, p2_hp_map),
    ):
        names = names + [None] * (TEAM_SIZE - len(names))
        values = {metric: [] for metric in DEPLOYED_METRICS}
        total_hp, weighted_level, deployed_levels = 0.0, 0.0, []
        for i, name in enumerate(names):
            if player == "p1":
                pokemon = p1_team[name]
            elif name == p2_lead["name"]:
                pokemon = p2_lead
            else:
                pokemon = {
                    "level": 100,
                    "base_hp": stats_map.get(name, {}).get("base_stats", {}).get("base_hp", 85 if name is None else 80),
                }
            level = pokemon["level"]
            turns = deployed_turns.get(name, 0)
            hp = calc_eff_stat(pokemon["base_hp"], level, is_hp=True)
            total_hp += hp
            weighted_level += level * turns
            if turns:
                deployed_levels.append(level)
            for metric, value in (
                ("deployed_turns", turns), ("level", level),
                ("remaining_effective_hp", hp * hp_map.get(name, {}).get("last_hp", 1.0)),
            ):
                values[metric].append(value)
                features[f"{player}_{i}_pkm_{metric}"] = value

        for metric, observations in values.items():
            for summary, value in (("sum", np.sum(observations)), ("mean", np.mean(observations)), ("std", np.std(observations, ddof=0))):
                features[f"{player}_{summary}_{metric}"] = float(value)
        features[f"{player}_deployed_count"] = len(deployed_turns)
        features[f"{player}_min_level"] = min(values["level"])
        features[f"{player}_underleveled_count"] = sum(level < 100 for level in values["level"])
        features[f"{player}_deployed_underleveled_count"] = sum(level < 100 for level in deployed_levels)
        features[f"{player}_deployed_min_lvl"] = min(deployed_levels, default=100)
        features[f"{player}_active_wmean_lvl"] = weighted_level / max(sum(deployed_turns.values()), 1)
        lead = p1_team.get(next(iter(deployed_turns), None), {}) if player == "p1" else p2_lead
        features[f"{player}_lead_level"] = lead.get("level", np.nan)
        features[f"{player}_rem_effective_hp_ratio"] = features[f"{player}_sum_remaining_effective_hp"] / total_hp if total_hp else 0.0

    for metric in DEPLOYED_METRICS:
        for summary in ("sum", "mean", "std"):
            features[f"diff_{summary}_{metric}"] = features[f"p1_{summary}_{metric}"] - features[f"p2_{summary}_{metric}"]
    for metric in DEPLOYED_SUMMARIES:
        features[f"diff_{metric}"] = features[f"p1_{metric}"] - features[f"p2_{metric}"]


# --- TACTICAL TEMPO, OFFENSIVE POWER, HAZARDS & REST ---
def get_tactics(turn: dict, p1_tactics: dict, p2_tactics: dict) -> None:
    """
    Record each turn's switches, moves, offensive power, RestTalk and phazing.
    Exclude replacements after observed KOs from voluntary switches.
    Offensive power is base power * STAB * type effectiveness * accuracy.
    Resolve Spikes/Rapid Spin for both players before recording hazard uptime
    """

    for player, opponent, tracker, opp_tracker in (
        ("p1", "p2", p1_tactics, p2_tactics),
        ("p2", "p1", p2_tactics, p1_tactics),
    ):
        state, opp_state = (turn[f"{side}_pokemon_state"] for side in (player, opponent))
        move = turn.get(f"{player}_move_details") or {}
        name = state["name"]
        fainted = state["status"] == "fnt" or state["hp_pct"] <= 0
        values = {metric: 0.0 for metric in TACTIC_METRICS}
        values["switches"] = int(tracker["prev_mon"] is not None and name != tracker["prev_mon"] and not tracker["prev_fainted"])
        tracker.update(prev_mon=name, prev_fainted=fainted)

        values["moves_used"] = int(bool(move))
        if move.get("category") in {"physical", "special"} and move.get("base_power", 0) > 0:
            attacker_types = stats_map.get(name, {}).get("types", ["notype", "notype"])
            defender_types = stats_map.get(opp_state["name"], {}).get("types", ["notype", "notype"])
            stab = STAB if move["type"] in attacker_types else 1.0
            values["eff_power"] = move["base_power"] * stab * move["accuracy"] * get_multiplier(move["type"], *defender_types)

        if not fainted:
            for metric, moves in TACTIC_MOVES.items():
                values[metric] = int(move.get("name") in moves)
            if move.get("name") == "spikes":
                tracker["spikes_active"] = 1
            elif move.get("name") == "rapidspin":
                opp_tracker["spikes_active"] = 0
        for metric, value in values.items():
            tracker["values"][metric].append(value)

    for tracker in (p1_tactics, p2_tactics):
        tracker["values"]["spikes_active"][-1] = tracker["spikes_active"]

def feature_tactics(p1_tactics: dict, p2_tactics: dict, n_turns: int, features: dict) -> None:
    """
    Add each tactical metric's sum, per-turn mean and population standard deviation.
    Include zero-action turns. Spikes sums count turns with that player's hazards up.
    Keep legacy names: moves/switches are rates, power is a mean, RestTalk/phazing
    are counts, and spikes_active is the final hazard flag. All diffs use P1 - P2
    """

    for player, tracker in (("p1", p1_tactics), ("p2", p2_tactics)):
        for metric in TACTIC_METRICS:
            values = tracker["values"][metric]
            total = sum(values)
            for summary, value in (("sum", total), ("mean", total / n_turns if n_turns else 0.0), ("std", np.std(values, ddof=0) if values else 0.0)):
                features[f"{player}_{summary}_{metric}"] = float(value)
            if metric in {"moves_used", "switches"}:
                features[f"{player}_{metric}"] = features[f"{player}_mean_{metric}"]
            elif metric == "eff_power":
                features[f"{player}_eff_power_mean"] = features[f"{player}_mean_eff_power"]
            elif metric == "spikes_active":
                features[f"{player}_{metric}"] = tracker["spikes_active"]
            else:
                features[f"{player}_{metric}"] = features[f"{player}_sum_{metric}"]

    for metric in TACTIC_METRICS:
        for summary in ("sum", "mean", "std"):
            features[f"diff_{summary}_{metric}"] = features[f"p1_{summary}_{metric}"] - features[f"p2_{summary}_{metric}"]
        legacy_metric = "eff_power_mean" if metric == "eff_power" else metric
        features[f"diff_{legacy_metric}"] = features[f"p1_{legacy_metric}"] - features[f"p2_{legacy_metric}"]


In [9]:
def extract_features(data: list) -> pd.DataFrame:
    """
    Extract comprehensive features combining absolute scale and differentials:
    1. base stats
    2. team weakness
    3. HP loss
    4. type multiplier
    5. statuses + p1 faint
    6. boosts
    7. effects
    8. last-turn snapshot
    10. p2 faint
    11. held items
    12. deployed Pokemon counts, levels and estimated remaining effective HP
    13. tactical tempo, effective offensive power, hazards, phazing & RestTalk
    """

    print("Extracting features...")
    feature_list = []
    for battle in data:
        features = {"battle_id": battle["battle_id"]}
        if "player_won" in battle:
            features["player_won"] = battle["player_won"]

        p1_hp_map, p2_hp_map = {}, {}
        p1_mults, p2_mults = ({mult: 0 for mult in MULTIPLIERS} for _ in range(2))
        p1_status, p2_status = {}, {}
        p1_boosts, p2_boosts = ({boost: 0 for boost in BOOSTS} for _ in range(2))
        p1_effects, p2_effects = {}, {}
        p1_last_seen, p2_last_seen = {}, {}
        p2_stats = {}
        p2_faint_tracker = {"confirmed": set(), "ghost": set()}
        p2_item_tracker = {"seen": {}, "prev_mon": None, "prev_hp": None, "prev_healing": False}
        p1_deployed_turns, p2_deployed_turns = {}, {}
        p1_tactics, p2_tactics = (
            {
                "prev_mon": None, "prev_fainted": False, "spikes_active": 0,
                "values": {metric: [] for metric in TACTIC_METRICS},
            }
            for _ in range(2)
        )

        timeline = battle["battle_timeline"]
        if not timeline:
            raise ValueError(f"Battle {battle['battle_id']} has no recorded turns.")
        for idx, turn in enumerate(timeline):
            get_stats(turn, p2_stats)
            get_hp_pct_lost(turn, p1_hp_map, p2_hp_map)
            get_mults(turn, p1_mults, p2_mults)
            get_statuses(turn, p1_status, p2_status)
            get_boosts(turn, p1_boosts, p2_boosts)
            get_effects(turn, p1_effects, p2_effects)
            get_last_seen(turn, p1_last_seen, p2_last_seen)
            get_p2_faint_evidence(turn, idx, timeline, p2_faint_tracker)
            get_items(turn, p2_item_tracker)
            get_deployed_turns(turn, p1_deployed_turns, p2_deployed_turns)
            get_tactics(turn, p1_tactics, p2_tactics)

        n_turns = len(timeline)
        # 1. base stats
        feature_base_stats(battle, p2_stats, features)
        # 2. team weakness
        feature_team_types(battle, p2_stats, features)
        # 3. HP loss
        feature_hp_lost(p1_hp_map, p2_hp_map, features)
        # 4. type multiplier
        feature_mults(p1_mults, p2_mults, features)
        # 5. statuses + p1 faint
        feature_statuses(p1_status, p2_status, n_turns, features)
        # 6. boosts
        feature_boosts(p1_boosts, p2_boosts, n_turns, features)
        # 7. effects
        feature_effects(p1_effects, p2_effects, n_turns, features)
        # 8. last-turn snapshot
        feature_last_seen(timeline[-1], p1_last_seen, p2_last_seen, features)
        # 10. p2 faint
        feature_p2_faint_evidence(p2_faint_tracker, p1_last_seen, p2_last_seen, features)
        # 11. held items
        feature_items(battle, p2_item_tracker, features)
        # 12. deployed Pokemon counts, levels and estimated remaining effective HP
        feature_deployed_and_levels(battle, p1_deployed_turns, p2_deployed_turns, p1_hp_map, p2_hp_map, features)
        # 13. tactical tempo, effective offensive power, hazards, phazing & RestTalk
        feature_tactics(p1_tactics, p2_tactics, n_turns, features)
        
        feature_list.append(features)

    print(f"SUCCESS: extracted {len(feature_list[0])} features.")
    return pd.DataFrame(feature_list).fillna(0)


# Extracting training and test features
train_df = extract_features(train_data)
print(f"Training features: {train_df.shape}.")
display(train_df.head())

test_df = extract_features(test_data)
print(f"Test features: {test_df.shape}.")
# display(test_df.head())

Extracting features...
SUCCESS: extracted 917 features.
Training features: (10000, 917).


,battle_id,player_won,p1_sum_bst,p1_mean_bst,p1_sum_hp,p1_mean_hp,p1_min_base_hp,p1_max_base_hp,p1_std_base_hp,p1_sum_atk,p1_mean_atk,p1_min_base_atk,p1_max_base_atk,p1_std_base_atk,p1_sum_def,p1_mean_def,p1_min_base_def,p1_max_base_def,p1_std_base_def,p1_sum_spa,p1_mean_spa,p1_min_base_spa,p1_max_base_spa,p1_std_base_spa,p1_sum_spd,p1_mean_spd,p1_min_base_spd,p1_max_base_spd,p1_std_base_spd,p1_sum_spe,p1_mean_spe,p1_min_base_spe,p1_max_base_spe,p1_std_base_spe,p1_std_base_bst,p2_sum_bst,p2_mean_bst,p2_sum_hp,p2_mean_hp,p2_min_base_hp,p2_max_base_hp,p2_std_base_hp,p2_sum_atk,p2_mean_atk,p2_min_base_atk,p2_max_base_atk,p2_std_base_atk,p2_sum_def,p2_mean_def,p2_min_base_def,p2_max_base_def,p2_std_base_def,p2_sum_spa,p2_mean_spa,p2_min_base_spa,p2_max_base_spa,p2_std_base_spa,p2_sum_spd,p2_mean_spd,p2_min_base_spd,p2_max_base_spd,p2_std_base_spd,p2_sum_spe,p2_mean_spe,p2_min_base_spe,p2_max_base_spe,p2_std_base_spe,p2_std_base_bst,diff_sum_hp,diff_mean_hp,diff_sum_atk,diff_mean_atk,diff_sum_def,diff_mean_def,diff_sum_spa,diff_mean_spa,diff_sum_spd,diff_mean_spd,diff_sum_spe,diff_mean_spe,diff_sum_bst,diff_mean_bst,diff_std_base_bst,p1_known_type_count,p1_type_diversity,p1_average_weakness_severity,p1_shared_weakness_type_count,p2_known_type_count,p2_type_diversity,p2_average_weakness_severity,p2_shared_weakness_type_count,diff_type_diversity,diff_average_weakness_severity,diff_shared_weakness_type_count,p1_sum_hp_lost,p1_hp_pct_lost_0_10,p1_hp_pct_lost_10_20,p1_hp_pct_lost_20_30,p1_mean_hp_pct_lost,p1_std_hp_pct_lost,p2_sum_hp_lost,p2_hp_pct_lost_0_10,p2_hp_pct_lost_10_20,p2_hp_pct_lost_20_30,p2_mean_hp_pct_lost,p2_std_hp_pct_lost,diff_hp_pct_lost_0_10,diff_hp_pct_lost_10_20,diff_hp_pct_lost_20_30,diff_sum_hp_pct_lost,diff_mean_hp_pct_lost,diff_std_hp_pct_lost,p1_count_mult,p1_sum_mult,p1_0_mult,p1_0.25_mult,p1_0.5_mult,p1_1_mult,p1_2_mult,p1_4_mult,p1_min_mult,p1_max_mult,p1_mean_mult,p1_std_mult,p2_count_mult,p2_sum_mult,p2_0_mult,p2_0.25_mult,p2_0.5_mult,p2_1_mult,p2_2_mult,p2_4_mult,p2_min_mult,p2_max_mult,p2_mean_mult,p2_std_mult,diff_count_mult,diff_0_mult,diff_0.25_mult,diff_0.5_mult,diff_1_mult,diff_2_mult,diff_4_mult,diff_min_mult,diff_max_mult,diff_sum_mult,diff_mean_mult,diff_std_mult,p1_sum_statuses,p1_slp,p1_status_frac_slp,p1_frz,p1_status_frac_frz,p1_tox,p1_status_frac_tox,p1_psn,p1_status_frac_psn,p1_par,p1_status_frac_par,p1_brn,p1_status_frac_brn,p1_mean_statuses,p2_sum_statuses,p2_slp,p2_status_frac_slp,p2_frz,p2_status_frac_frz,p2_tox,p2_status_frac_tox,p2_psn,p2_status_frac_psn,p2_par,p2_status_frac_par,p2_brn,p2_status_frac_brn,p2_mean_statuses,diff_slp,diff_status_frac_slp,diff_frz,diff_status_frac_frz,diff_tox,diff_status_frac_tox,diff_psn,diff_status_frac_psn,diff_par,diff_status_frac_par,diff_brn,diff_status_frac_brn,diff_sum_statuses,diff_mean_statuses,p1_sum_boosts,p1_atk_boost,p1_def_boost,p1_spa_boost,p1_spd_boost,p1_spe_boost,p1_mean_boosts,p2_sum_boosts,p2_atk_boost,p2_def_boost,p2_spa_boost,p2_spd_boost,p2_spe_boost,p2_mean_boosts,diff_atk_boost,diff_def_boost,diff_spa_boost,diff_spd_boost,diff_spe_boost,diff_sum_boosts,diff_mean_boosts,p1_sum_good_effects,p1_beatup_effect,p1_bide_effect,p1_destinybond_effect,p1_endure_effect,p1_focusband_effect,p1_focusenergy_effect,p1_lockon_effect,p1_magnitude_effect,p1_mimic_effect,p1_mindreader_effect,p1_mysteryberry_effect,p1_protect_effect,p1_quickclaw_effect,p1_safeguard_effect,p1_substitute_effect,p1_typechange_effect,p1_mean_good_effects,p1_sum_bad_effects,p1_attract_effect,p1_bind_effect,p1_clamp_effect,p1_confusion_effect,p1_curse_effect,p1_disable_effect,p1_encore_effect,p1_firespin_effect,p1_foresight_effect,p1_futuresight_effect,p1_leechseed_effect,p1_nightmare_effect,p1_perish_effect,p1_pursuit_effect,p1_spite_effect,p1_trapped_effect,p1_whirlpool_effect,p1_wrap_effect,p1_mean_bad_effects,p1_sum_effects,p1_mean_effects,p2_sum_good_effects,p2_beatup_effect,p2_bide_effect,p2_destinybond_effect,p2_endure_effect,p2_focusband_effect,p2_focusenergy_effec

Extracting features...
SUCCESS: extracted 916 features.
Test features: (5000, 916).


# 4. Build Models

Define preprocessing pipelines for logistic regression, LightGBM, XGBoost, CatBoost, and a soft-voting ensemble.

In [10]:
def make_model_pipeline(estimator, categorical_features: list):
    """
    Fit encoding, numeric imputation and scaling within each training fold
    """

    preprocessing = ColumnTransformer(
        [("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)],
        remainder=SimpleImputer(strategy="median", keep_empty_features=True),
        verbose_feature_names_out=False,
    )
    
    scaler = StandardScaler().set_output(transform="pandas")
    return make_pipeline(preprocessing, scaler, estimator)

def make_lgb(random_state: int = RANDOM_STATE):
    """
    Build a LightGBM classifier
    """

    return LGBMClassifier(
            n_estimators=ENSEMBLE_N_ESTIMATORS,
            max_depth=5,
            learning_rate=0.04,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            verbose=-1,
            random_state=random_state,
        )

def make_xgb(random_state: int = RANDOM_STATE):
    """
    Build an XGBoost classifier
    """

    return XGBClassifier(
            n_estimators=ENSEMBLE_N_ESTIMATORS,
            max_depth=5,
            learning_rate=0.04,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=random_state,
            n_jobs=-1,
        )

def make_cat(random_state: int = RANDOM_STATE):
    """
    Build a CatBoost classifier
    """

    return CatBoostClassifier(
            iterations=ENSEMBLE_N_ESTIMATORS,
            depth=5,
            learning_rate=0.04,
            loss_function="Logloss",
            verbose=False,
            allow_writing_files=False,
            random_seed=random_state,
            thread_count=1,
        )

def make_lr(random_state: int = RANDOM_STATE):
    """
    Build a logistic regression classifier
    """
    
    return LogisticRegression(
        max_iter=1000, 
        C=1.0, 
        random_state=random_state
    )
    

def make_ensemble(random_state: int = RANDOM_STATE) -> VotingClassifier:
    """
    Build the four-model soft-voting ensemble used by the reference ru
    """

    estimators = [
        ("lgb", make_lgb(random_state)),
        ("xgb", make_xgb(random_state)),
        ("cat", make_cat(random_state)),
        ("lr",  make_lr(random_state)),
    ]

    return VotingClassifier(
        estimators=estimators,
        voting="soft",
        weights=ENSEMBLE_WEIGHTS,
    )

def build_model(model_name: str, categorical_features: list, random_state: int = RANDOM_STATE):
    """
    Build one standalone model or the complete soft-voting ensemble
    """
    
    model_name = model_name.strip().lower()
    match model_name:
        case "ensemble":
            estimator = make_ensemble(random_state)
        case "lgb":
            estimator = make_lgb(random_state)
        case "xgb":
            estimator = make_xgb(random_state)
        case "cat":
            estimator = make_cat(random_state)
        case "lr":
            estimator = make_lr(random_state)
        case _:
            raise ValueError("Unknown model. Choose: ensemble, lgb, xgb, cat, lr.")

    return make_model_pipeline(estimator, categorical_features)

## 4.1 Computer Performance Metrics

Measure accuracy, precision, recall, F1, and ROC-AUC using predicted labels and win probabilities.

In [11]:
def compute_metrics(y_true: pd.Series | np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> dict:
    """
    Return accuracy, precision, recall, F1, and ROC-AUC for binary predictions.
    y_true and y_pred contain 0/1 labels.
    y_prob contains probabilities of class 1
    """

    return {
        # (TP + TN) / total
        "accuracy": accuracy_score(y_true, y_pred),
        # TP / (TP + FP)
        "precision": precision_score(y_true, y_pred, zero_division=0),
        # TP / (TP + FN)
        "recall": recall_score(y_true, y_pred, zero_division=0),
        # 2 * (precision * recall) / (precision + recall) = 2 * TP / (2 * TP + FP + FN)
        "f1": f1_score(y_true, y_pred, zero_division=0),
        # Area Under the Receiver Operating Characteristic Curve
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

## 4.2 Select Features

Select the configured subset (`FEATURES_DANI`, `FEATURES_FIL`, or `FEATURES_LORE`) and align training and test columns in the same order. Remove duplicate feature names, check for missing columns, and normalize categorical values. Keep `battle_id` out of the predictors and use `player_won` only as the training target.

In [12]:
def prepare_selected_features(train_df: pd.DataFrame, test_df: pd.DataFrame, feature_list: list) -> tuple:
    """
    Align features and normalize categorical values without using labels
    """

    # Remove duplicates while preserving order
    feature_cols = list(dict.fromkeys(feature_list))
    if not feature_cols:                                        raise ValueError("ALLOWED_FEATURES must contain at least one predictor.")
    if {"battle_id", "player_won"}.intersection(feature_cols):  raise ValueError("Identifiers and target labels cannot be predictors.")

    # for each frame, check that all feature columns are present
    for name, frame in (("training", train_df), ("test", test_df)):
        missing = sorted(set(feature_cols) - set(frame.columns))
        if missing:
            raise ValueError(f"Missing {name} features: {missing}")

    # extract X, y and X_test
    X = train_df.loc[:, feature_cols].copy()
    X_test = test_df.loc[:, feature_cols].copy()
    y = train_df["player_won"] 
    
    # checks that y is binary and contains no NaN values
    if y.isna().any() or not y.isin([0, 1]).all():
        raise ValueError("player_won must contain only 0 and 1.")
    y = y.astype(int)

    categorical_features = [column for column in feature_cols if not pd.api.types.is_numeric_dtype(X[column])]
    for frame in (X, X_test):
        for column in feature_cols:
            if column in categorical_features:
                frame[column] = frame[column].astype("string").fillna("__unknown__").astype(str)
            else:
                frame[column] = pd.to_numeric(frame[column], errors="raise").replace([np.inf, -np.inf], np.nan)
    print(f"Selected features: {len(feature_cols)}; categorical: {len(categorical_features)}")
    return X, y, X_test, categorical_features

# 5. Train

Use repeated stratified cross-validation to summarize model performance and collect out-of-fold win probabilities.

In [13]:
def train_and_evaluate_cv(
    model_name: str, X: pd.DataFrame, y: pd.Series, categorical_features: list,
    k: int = K_FOLDS, n_repeats: int = N_REPEATS, random_state: int = RANDOM_STATE,
) -> tuple:
    """
    Evaluate one model ("ensemble", "lr", ...) with repeated stratified CV.
    For the ensemble, each voter is also evaluated separately.
    """

    splitter = RepeatedStratifiedKFold(n_splits=k, n_repeats=n_repeats, random_state=random_state)

    # The main model has no prefix in the records; the voters are prefixed with their name
    if model_name == "ensemble":    model_names = [model_name, "lgb", "xgb", "cat", "lr"]
    else:                           model_names = [model_name]

    # Initialize out-of-fold predictions and counts for each model
    oof_probs = {name: np.zeros(len(X)) for name in model_names}
    oof_counts = np.zeros(len(X), dtype=int)
    records = []

    print(f"Starting {n_repeats} x {k}-fold validation for '{model_name}' ({X.shape[1]} features).")
    for i, (train_idx, val_idx) in enumerate(splitter.split(X, y)):
        repeat, fold = divmod(i, k)
        model = build_model(model_name, categorical_features, random_state=random_state + i)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        fitted_models = {model_name: model}


        if model_name == "ensemble":
            # Voters are fitted on the preprocessed data, so they need the transformed features
            voting_model = model.named_steps["votingclassifier"]
            X_train_transformed = model[:-1].transform(X.iloc[train_idx])
            X_val_transformed = model[:-1].transform(X.iloc[val_idx])
            fitted_models.update(voting_model.named_estimators_)
        record = {"repeat": repeat + 1, "fold": fold + 1, "feature_count": X.shape[1]}
        
        for name, estimator in fitted_models.items():
            if name == model_name:
                train_prob = estimator.predict_proba(X.iloc[train_idx])[:, 1]
                val_prob = estimator.predict_proba(X.iloc[val_idx])[:, 1]
            else:
                train_prob = estimator.predict_proba(X_train_transformed)[:, 1]
                val_prob = estimator.predict_proba(X_val_transformed)[:, 1]
            train_metrics = compute_metrics(
                y.iloc[train_idx], (train_prob >= DECISION_THRESHOLD).astype(int), train_prob,
            )
            val_metrics = compute_metrics(
                y.iloc[val_idx], (val_prob >= DECISION_THRESHOLD).astype(int), val_prob,
            )
            prefix = "" if name == model_name else f"{name}_"
            record.update({f"{prefix}train_{metric}": value for metric, value in train_metrics.items()})
            record.update({f"{prefix}val_{metric}": value for metric, value in val_metrics.items()})
            oof_probs[name][val_idx] += val_prob
        oof_counts[val_idx] += 1
        records.append(record)
        print(
            f"Repeat {repeat + 1}/{n_repeats} | Fold {fold + 1}/{k} | "
            f"Train accuracy: {record['train_accuracy']:.4f} | "
            f"Validation accuracy: {record['val_accuracy']:.4f} | ROC-AUC: {record['val_roc_auc']:.4f}"
        )

    if not np.all(oof_counts == n_repeats):
        raise RuntimeError("Every battle must receive one OOF prediction per repeat.")
    cv_results = pd.DataFrame(records)
    summaries = {}
    for name in model_names:
        oof_probs[name] /= oof_counts
        oof_metrics = compute_metrics(y, (oof_probs[name] >= DECISION_THRESHOLD).astype(int), oof_probs[name])
        prefix = "" if name == model_name else f"{name}_"
        summaries[name] = pd.DataFrame({
            "train_mean": {metric: cv_results[f"{prefix}train_{metric}"].mean() for metric in oof_metrics},
            "validation_mean": {metric: cv_results[f"{prefix}val_{metric}"].mean() for metric in oof_metrics},
            "validation_std": {metric: cv_results[f"{prefix}val_{metric}"].std(ddof=0) for metric in oof_metrics},
            "oof": oof_metrics,
        })
    summary = pd.concat(summaries, names=["model", "metric"])
    print(f"\nValidation summary for '{model_name}' (accuracy is the competition metric):")
    display(summary)
    return cv_results, oof_probs, summary

# 6. Submission File

Match predicted labels to `battle_id` values in the sample submission, preserving its row order. Check that every battle has a prediction, then export only `battle_id` and the integer `player_won` label to CSV.

In [14]:
def predict_and_save(predictions: np.ndarray, test_battle_ids: pd.Series, output_path: str, sample_path: str = SAMPLE_SUBMISSION_PATH) -> None:
    """
    Format and save the submission file using the required template
    """
    
    sample_submission = pd.read_csv(sample_path)
    expected_columns = ["battle_id", "player_won"]
    if list(sample_submission.columns) != expected_columns:
        raise ValueError(f"ERROR: expected submission columns {expected_columns}, got {list(sample_submission.columns)}")

    predictions_by_id = pd.Series(predictions, index=test_battle_ids)
    sample_submission["player_won"] = sample_submission["battle_id"].map(predictions_by_id)

    if sample_submission["player_won"].isna().any():
        raise ValueError(f"ERROR: some battle IDs in the '{sample_path}' have no matching prediction.")

    sample_submission["player_won"] = sample_submission["player_won"].astype(int)
    sample_submission.to_csv(output_path, index=False)

    print(f"SUCCESS: file '{output_path}' created.")

# 7. Main

Run the final cell to process the three entries in `MODEL_CONFIGS`. For each entry, evaluate its model and feature subset with 3 repeats of 5-fold stratified cross-validation, then refit on all training battles. Convert test win probabilities to labels using `DECISION_THRESHOLD` (0.5) and save the corresponding file:

- `submission_lr_dani.csv`: logistic regression using `FEATURES_DANI`.
- `submission_lr_fil.csv`: logistic regression using `FEATURES_FIL`.
- `submission_ens_lore.csv`: the LightGBM, XGBoost, CatBoost, and logistic regression soft-voting ensemble using `FEATURES_LORE`.

Each file is written after its model finishes validation and final fitting, in the notebook's current working directory.

In [15]:
"""
For each configured model: repeated stratified CV, final fit on the whole train set,
and submission CSV. Every model sees the same folds (same random_state and same y).
"""

trained_models, cv_outputs = {}, {}
for label, config in MODEL_CONFIGS.items():
    model_name, feature_list, output_path = config
    print(f"\n{'=' * 40} {label} {'=' * 40}")
    X_model, y_model, X_test_model, categorical_model = prepare_selected_features(train_df, test_df, feature_list)

    # Cross-validation outputs. It contains the results for each model lr_dani, lr_fil, ens_lore
    cv_outputs[label] = train_and_evaluate_cv(model_name, X_model, y_model, categorical_model)

    # Final fit on all the training battles + submission
    model = build_model(model_name, categorical_model)
    model.fit(X_model, y_model)

    probabilities = model.predict_proba(X_test_model)[:, 1]
    predictions = (probabilities >= DECISION_THRESHOLD).astype(int)
    predict_and_save(
        predictions,
        test_df["battle_id"],
        output_path=output_path,
        sample_path=SAMPLE_SUBMISSION_PATH,
    )
    trained_models[label] = model


======================================== lr_dani ========================================
Selected features: 76; categorical: 0
Starting 3 x 5-fold validation for 'lr' (76 features).
Repeat 1/3 | Fold 1/5 | Train accuracy: 0.6921 | Validation accuracy: 0.6800 | ROC-AUC: 0.7518
Repeat 1/3 | Fold 2/5 | Train accuracy: 0.6904 | Validation accuracy: 0.6870 | ROC-AUC: 0.7514
Repeat 1/3 | Fold 3/5 | Train accuracy: 0.6909 | Validation accuracy: 0.6945 | ROC-AUC: 0.7504
Repeat 1/3 | Fold 4/5 | Train accuracy: 0.6885 | Validation accuracy: 0.6895 | ROC-AUC: 0.7599
Repeat 1/3 | Fold 5/5 | Train accuracy: 0.6905 | Validation accuracy: 0.6820 | ROC-AUC: 0.7442
Repeat 2/3 | Fold 1/5 | Train accuracy: 0.6947 | Validation accuracy: 0.6675 | ROC-AUC: 0.7378
Repeat 2/3 | Fold 2/5 | Train accuracy: 0.6919 | Validation accuracy: 0.6860 | ROC-AUC: 0.7483
Repeat 2/3 | Fold 3/5 | Train accuracy: 0.6875 | Validation accuracy: 0.6920 | ROC-AUC: 0.7535
Repeat 2/3 | Fold 4/5 | Train accuracy: 0.6881 | Validat

train_mean  validation_mean  validation_std       oof
model metric                                                          
lr    accuracy     0.690800         0.686067        0.008380  0.686200
      precision    0.686590         0.682466        0.008072  0.682406
      recall       0.702083         0.696000        0.013726  0.696600
      f1           0.694246         0.689112        0.009367  0.689430
      roc_auc      0.758865         0.750588        0.006050  0.751141

SUCCESS: file '/kaggle/working/submission_lr_dani.csv' created.

======================================== lr_fil ========================================
Selected features: 84; categorical: 0
Starting 3 x 5-fold validation for 'lr' (84 features).
Repeat 1/3 | Fold 1/5 | Train accuracy: 0.6944 | Validation accuracy: 0.6820 | ROC-AUC: 0.7524
Repeat 1/3 | Fold 2/5 | Train accuracy: 0.6894 | Validation accuracy: 0.6865 | ROC-AUC: 0.7517
Repeat 1/3 | Fold 3/5 | Train accuracy: 0.6893 | Validation accuracy: 0.6905 | ROC-AUC: 0.7506
Repeat 1/3 | Fold 4/5 | Train accuracy: 0.6913 | Validation accuracy: 0.6885 | ROC-AUC: 0.7604
Repeat 1/3 | Fold 5/5 | Train accuracy: 0.6899 | Validation accuracy: 0.6830 | ROC-AUC: 0.7427
Repeat 2/3 | Fold 1/5 | Train accuracy: 0.6945 | Validation accuracy: 0.6695 | ROC-AUC: 0.7388
Repeat 2/3 | Fold 2/5 | Train accuracy: 0.6917 | Validation accuracy: 0.6840 | ROC-AUC: 0.7487
Repeat 2/3 | Fold 3/5 | Train accuracy: 0.6894 | Validation accuracy: 0.6925 | ROC-AUC: 

train_mean  validation_mean  validation_std       oof
model metric                                                          
lr    accuracy     0.691092         0.684767        0.007108  0.686100
      precision    0.687274         0.681258        0.007604  0.682702
      recall       0.701300         0.694600        0.011424  0.695400
      f1           0.694211         0.687812        0.007577  0.688992
      roc_auc      0.759319         0.750707        0.006157  0.751257

SUCCESS: file '/kaggle/working/submission_lr_fil.csv' created.

======================================== ens_lore ========================================
Selected features: 119; categorical: 3
Starting 3 x 5-fold validation for 'ensemble' (119 features).
Repeat 1/3 | Fold 1/5 | Train accuracy: 0.7556 | Validation accuracy: 0.6775 | ROC-AUC: 0.7572
Repeat 1/3 | Fold 2/5 | Train accuracy: 0.7552 | Validation accuracy: 0.6955 | ROC-AUC: 0.7543
Repeat 1/3 | Fold 3/5 | Train accuracy: 0.7558 | Validation accuracy: 0.6835 | ROC-AUC: 0.7538
Repeat 1/3 | Fold 4/5 | Train accuracy: 0.7522 | Validation accuracy: 0.6835 | ROC-AUC: 0.7597
Repeat 1/3 | Fold 5/5 | Train accuracy: 0.7595 | Validation accuracy: 0.6815 | ROC-AUC: 0.7491
Repeat 2/3 | Fold 1/5 | Train accuracy: 0.7570 | Validation accuracy: 0.6750 | ROC-AUC: 0.7448
Repeat 2/3 | Fold 2/5 | Train accuracy: 0.7572 | Validation accuracy: 0.6835 | ROC-AUC: 0.7510
Repeat 2/3 | Fold 3/5 | Train accuracy: 0.7571 | Validation accuracy: 0.6890 | 

train_mean  validation_mean  validation_std       oof
model    metric                                                          
ensemble accuracy     0.756775         0.682267        0.006126  0.682500
         precision    0.749107         0.678084        0.005875  0.678118
         recall       0.772167         0.694067        0.011012  0.694800
         f1           0.760459         0.685942        0.007013  0.686358
         roc_auc      0.843171         0.753598        0.005284  0.754489
lgb      accuracy     0.798967         0.679233        0.005310  0.682100
         precision    0.791599         0.675303        0.006047  0.677936
         recall       0.811617         0.690600        0.010455  0.693800
         f1           0.801478         0.682813        0.005947  0.685776
         roc_auc      0.887644         0.748899        0.004377  0.752027
xgb      accuracy     0.805158         0.679933        0.008970  0.681600
         precision    0.798009         0.676407        0.009612  0.677830
         recall       0.817183         0.690133        0.010689  0.692200
         f1           0.807470         0.683160        0.008645  0.684940
         roc_auc      0.893988         0.749511        0.005510  0.752509
cat      accuracy     0.710500         0.677500        0.006979  0.679500
         precision    0.703875         0.672823        0.007669  0.674713
         recall       0.726750         0.691200        0.009282  0.693200
         f1           0.715121         0.681850        0.006841  0.683832
         roc_auc      0.788333         0.747985        0.005092  0.749029
lr       accuracy     0.690883         0.680067        0.007026  0.680400
         precision    0.687045         0.676670        0.007247  0.676586
         recall       0.701150         0.689800        0.012189  0.691200
         f1           0.694022         0.683117        0.007870  0.683815
         roc_auc      0.764171         0.748235        0.007143  0.749160

SUCCESS: file '/kaggle/working/submission_ens_lore.csv' created.
